# Week 45, Day 6: Forge: Week 45 challenge, quiz and deep read

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w045/d6.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
import numpy as np
print("1.", round(1 / (1 + np.exp(-2)), 3))
print("2.", round(np.tanh(1.0), 3), "s")
print("3.", 1.0, "s")

### Exercise: Part A: fitting both curves

Write pred(p), returning predicted accuracy and mean RT at each coherence for p = (k, a, T_nd), using a^2 for the zero-drift mean decision time. Fit with Nelder-Mead on the summed squared errors, each scaled by 0.0004.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
r = np.random.default_rng(2005)
cohs = np.array([0, 0.032, 0.064, 0.128, 0.256, 0.512])

def sim(k, a, ter, n=600, dt=0.001, eta=0.0):
    acc, mrt, out = [], [], []
    for c in cohs:
        v = k * c + eta * r.normal(size=n); x = np.zeros(n); done = np.zeros(n, bool); ch = np.zeros(n); rt = np.zeros(n); i = 0
        while not done.all() and i < 10000:
            idx = ~done; x[idx] += v[idx] * dt + np.sqrt(dt) * r.normal(size=idx.sum()); i += 1
            hit = idx & (np.abs(x) >= a); ch[hit] = np.sign(x[hit]); rt[hit] = i * dt + ter; done |= hit
        cor = np.where(c == 0, r.random(n) < 0.5, ch > 0)
        acc.append(cor.mean()); mrt.append(rt.mean()); out.append((cor, rt))
    return np.array(acc), np.array(mrt), out

acc, mrt, _ = sim(10.0, 0.8, 0.35)

def pred(p):
    k, a, ter = p
    v = k * cohs
    ...
    return P, M

def loss(p):
    P, M = pred(p)
    return np.sum((P - acc) ** 2 / 0.0004) + np.sum((M - mrt) ** 2 / 0.0004)

res = minimize(loss, [5, 1, 0.2], method="Nelder-Mead", options={"maxiter": 2000, "xatol": 1e-4, "fatol": 1e-6})
print("fitted k, a, T_nd:", np.round(res.x, 3))
P, M = pred(res.x)
fig, ax = plt.subplots(1, 2, figsize=(9, 3.5))
ax[0].plot(cohs, acc, "o"); ax[0].plot(cohs, P); ax[0].set_ylabel("accuracy")
ax[1].plot(cohs, mrt, "o"); ax[1].plot(cohs, M); ax[1].set_ylabel("mean RT (s)")
for a_ in ax: a_.set_xlabel("coherence")
plt.tight_layout(); plt.show()

In [ ]:
import numpy as np
def test_fit():
    "the fit recovers k within 1.5, a within 0.08 and T_nd within 0.03 s"
    k_, a_, t_ = res.x
    assert abs(k_ - 10) < 1.5 and abs(a_ - 0.8) < 0.08 and abs(t_ - 0.35) < 0.03
def test_curves():
    "the fitted curves pass within 0.05 of every accuracy and 60 ms of every mean RT"
    P, M = pred(res.x)
    assert np.abs(P - acc).max() < 0.05 and np.abs(M - mrt).max() < 0.06

test_fit()
test_curves()
print("All checks passed")

### Exercise: Part B: drift variability

Simulate with eta = 0 and eta = 1.5. For coherences 6.4% and 12.8% (indices 2 and 3), compute mean error RT minus mean correct RT.

In [ ]:
_, _, out0 = sim(10.0, 0.8, 0.35, n=1500)
_, _, out_eta = sim(10.0, 0.8, 0.35, n=1500, eta=1.5)

def gaps(out):
    return [...]

gap0, gap_eta = gaps(out0), gaps(out_eta)
for i in (2, 3):
    print(f"{100 * cohs[i]:.1f}%: error minus correct RT, no variability {1000 * gap0[i]:+.0f} ms, with variability {1000 * gap_eta[i]:+.0f} ms")

In [ ]:
import numpy as np
def test_none():
    "without drift variability, error and correct mean RTs differ by less than 40 ms"
    assert all(abs(gap0[i]) < 0.04 for i in (2, 3))
def test_eta():
    "with drift variability, errors are at least 50 ms slower than correct responses"
    assert all(gap_eta[i] > 0.05 for i in (2, 3))

test_none()
test_eta()
print("All checks passed")